In [28]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import sys, os
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

ROOT = os.path.abspath("..")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
from src.utils.preprocessing import wrangle_data, load_file
from sklearn.model_selection import train_test_split
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_curve
from src.models.evaluate import evaluate, evaluate_anomaly, identify_best_model, print_final_results

In [29]:
transactions_file_path = os.getenv("transactions_file_path")
transaction_dataframe= load_file(transactions_file_path)

In [30]:
transaction_dataframe.drop(columns=["TX_ID", "ALERT_ID", "TX_TYPE"], inplace=True)

In [31]:
transaction_dataframe.info()

<class 'pandas.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 5 columns):
 #   Column               Non-Null Count    Dtype  
---  ------               --------------    -----  
 0   SENDER_ACCOUNT_ID    1048575 non-null  int64  
 1   RECEIVER_ACCOUNT_ID  1048575 non-null  int64  
 2   TX_AMOUNT            1048575 non-null  float64
 3   TIMESTAMP            1048575 non-null  int64  
 4   IS_FRAUD             1048575 non-null  bool   
dtypes: bool(1), float64(1), int64(3)
memory usage: 33.0 MB


In [32]:
#sort values by TIMESTAMP and Rrbuilds the row numbers (index) starting from
transaction_dataframe = transaction_dataframe.sort_values("TIMESTAMP").reset_index(drop=True)

In [33]:
transaction_dataframe["TIMESTAMP"].tail(10)

1048565    158
1048566    158
1048567    158
1048568    158
1048569    158
1048570    158
1048571    158
1048572    158
1048573    158
1048574    158
Name: TIMESTAMP, dtype: int64

In [34]:
transaction_dataframe["SENDER_STEP_COUNT"] = transaction_dataframe.groupby(["SENDER_ACCOUNT_ID", "TIMESTAMP"])["TX_AMOUNT"].transform("count")
transaction_dataframe["RECEIVER_STEP_COUNT"] = transaction_dataframe.groupby(["RECEIVER_ACCOUNT_ID", "TIMESTAMP"])["TX_AMOUNT"].transform("count")
transaction_dataframe["SENDER_RECEIVER_PAIR_COUNT"] = transaction_dataframe.groupby(["SENDER_ACCOUNT_ID", "RECEIVER_ACCOUNT_ID"])["TX_AMOUNT"].transform("count")

In [35]:
transaction_dataframe["SENDER_STEP_COUNT"].value_counts()

SENDER_STEP_COUNT
1     344144
8      66568
9      56934
10     48440
11     47564
7      42483
14     41160
13     39494
6      39126
17     35292
5      30775
12     27204
20     23760
21     23730
18     15390
34     12954
22     12870
33     11946
35     10395
36      8640
38      8436
37      8251
3       8220
32      8096
39      8034
40      7720
19      7657
15      7395
43      6837
46      6624
42      5292
45      4995
2       4546
44      4224
41      3854
4       3504
16      1776
48      1536
23      1357
31       992
24       360
Name: count, dtype: int64

In [36]:
# prepare features and target
X = transaction_dataframe.drop(columns=["IS_FRAUD","TIMESTAMP"])
y = transaction_dataframe["IS_FRAUD"]

In [37]:
#temporary
#X = transaction_dataframe.drop(columns=["SENDER_TOTAL_COUNT"])

In [38]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 6 columns):
 #   Column                      Non-Null Count    Dtype  
---  ------                      --------------    -----  
 0   SENDER_ACCOUNT_ID           1048575 non-null  int64  
 1   RECEIVER_ACCOUNT_ID         1048575 non-null  int64  
 2   TX_AMOUNT                   1048575 non-null  float64
 3   SENDER_STEP_COUNT           1048575 non-null  int64  
 4   RECEIVER_STEP_COUNT         1048575 non-null  int64  
 5   SENDER_RECEIVER_PAIR_COUNT  1048575 non-null  int64  
dtypes: float64(1), int64(5)
memory usage: 48.0 MB


In [39]:
cutoff = int(len(transaction_dataframe) * 0.8)
X_train_full, y_train_full = X.iloc[: cutoff], y.iloc[:cutoff]
X_test, y_test =  X.iloc[cutoff: ], y.iloc[cutoff:]

cutoff_train_val = int(len(X_train_full) * 0.8)
X_train, y_train = X_train_full.iloc[:cutoff_train_val], y_train_full.iloc[:cutoff_train_val]
X_validation, y_validation = X_train_full.iloc[cutoff_train_val:], y_train_full.iloc[cutoff_train_val:]


In [ ]:
#fraud rate from train dataset
fraud_rate = y_train.value_counts(normalize=True).iloc[1]
print(fraud_rate)

0.0012674343752235176


In [ ]:
int(y_train.mean())

np.float64(0.001330674963641132)

In [41]:
cutoff_train_val

671088

In [42]:
y_train_full

0         False
1         False
2         False
3         False
4         False
          ...  
838855    False
838856    False
838857    False
838858    False
838859    False
Name: IS_FRAUD, Length: 838860, dtype: bool

In [43]:
#split data into train, validation and test
#X_train_full, X_test, y_train_full, y_test = train_test_split(
       # X, y, test_size=0.2, stratify=y, random_state=42
  #  )

#X_train, X_validation, y_train, y_validation = train_test_split(
       # X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
    #)

In [44]:
logistic_regression_pipeline = Pipeline([
        ("smote",  SMOTE(random_state=42)),
        ("scaler", StandardScaler()),
        ("model",  LogisticRegression(max_iter=1000, random_state=42)),
    ])
logistic_regression_pipeline.fit(X_train, y_train)

,steps,"[('smote', ...), ('scaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[bool](2,)","[False, True]"
feature_names_in_,"ndarray[object](6,)","['SENDER_ACCOUNT_ID','RECEIVER_ACCOUNT_ID','TX_AMOUNT','SENDER_STEP_COUNT', 'RECEIVER_STEP_COUNT','SENDER_RECEIVER_PAIR_COUNT']"
n_features_in_,int,6
,random_state,42
,sampling_strategy,'auto'
,k_neighbors,5


In [45]:
X_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 671088 entries, 0 to 671087
Data columns (total 6 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   SENDER_ACCOUNT_ID           671088 non-null  int64  
 1   RECEIVER_ACCOUNT_ID         671088 non-null  int64  
 2   TX_AMOUNT                   671088 non-null  float64
 3   SENDER_STEP_COUNT           671088 non-null  int64  
 4   RECEIVER_STEP_COUNT         671088 non-null  int64  
 5   SENDER_RECEIVER_PAIR_COUNT  671088 non-null  int64  
dtypes: float64(1), int64(5)
memory usage: 30.7 MB


In [46]:
random_forest_pipeline = Pipeline([
        ("smote", SMOTE(random_state=42)),
        ("model", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ])
random_forest_pipeline.fit(X_train, y_train)

,steps,"[('smote', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[bool](2,)","[False, True]"
feature_names_in_,"ndarray[object](6,)","['SENDER_ACCOUNT_ID','RECEIVER_ACCOUNT_ID','TX_AMOUNT','SENDER_STEP_COUNT', 'RECEIVER_STEP_COUNT','SENDER_RECEIVER_PAIR_COUNT']"
n_features_in_,int,6
,random_state,42
,sampling_strategy,'auto'
,k_neighbors,5


In [47]:
# 1. Get probabilities for fraud (Class 1)
probabilities = random_forest_pipeline.predict_proba(X_test)[:, 1]

# 2. Calculate the curve points
precisions, recalls, thresholds = precision_recall_curve(y_test, probabilities)

In [48]:
xgboost_pipeline =   Pipeline([
    ("smote", SMOTE(random_state=42)),
    ("model", XGBClassifier(random_state=42, eval_metric="logloss", n_jobs=-1))])
xgboost_pipeline.fit(X_train, y_train)

,steps,"[('smote', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[int64](2,)","[0,1]"
feature_names_in_,"ndarray[object](6,)","['SENDER_ACCOUNT_ID','RECEIVER_ACCOUNT_ID','TX_AMOUNT','SENDER_STEP_COUNT', 'RECEIVER_STEP_COUNT','SENDER_RECEIVER_PAIR_COUNT']"
n_features_in_,int,6
,random_state,42
,sampling_strategy,'auto'
,k_neighbors,5


In [49]:
len(thresholds)

29

In [68]:

CONTAMINATION = round(fraud_rate,4)
isolation_forest = IsolationForest(contamination=CONTAMINATION, random_state=42, n_jobs=-1,)
isolation_forest.fit(X_train)

,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",np.float64(0.0013)
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary <n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary <warm_start>`... versionadded:: 0.21",False
Name,Type,Value
estimator_ estimator_: :class:`~sklearn.tree.ExtraTreeRegressor` instanceThe child estimator template used to create the collection offitted sub-estimators... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,ExtraTreeRegressor,ExtraTreeRegr...ndom_state=42)


In [50]:
# Find the threshold where we catch at least 80% of fraud (Recall >= 0.80)
target_recall = 0.80
idx = np.where(recalls >= target_recall)[0][-1] # Get the closest match

best_threshold = thresholds[idx]
print(f"To catch 80% of fraud, use threshold: {best_threshold:.4f}")

To catch 80% of fraud, use threshold: 0.9900


In [69]:
results=[]
results.append(evaluate(logistic_regression_pipeline, X_validation, y_validation, "Logistic Regression"))
results.append(evaluate(random_forest_pipeline, X_validation, y_validation, "Random Forest"))
results.append(evaluate(xgboost_pipeline, X_validation, y_validation, "XGBoost"))
results.append(evaluate_anomaly(isolation_forest, X_validation, y_validation, "Isolation Forest"))

In [70]:

overall_best_classifier_model_name = identify_best_model(results)



── Model Comparison ─────────────────────────────────────
                     precision  recall  f1_score     fpr     fnr  roc_auc  pr_auc
model                                                                            
Logistic Regression     0.0405  0.9568    0.0776  0.0219  0.0432   0.9809  0.8460
Random Forest           1.0000  0.9877    0.9938  0.0000  0.0123   1.0000  0.9962
XGBoost                 1.0000  0.9877    0.9938  0.0000  0.0123   0.9987  0.9938
Isolation Forest        0.0000  0.0000    0.0000  0.0091  1.0000   0.6896  0.0014

── Best Model ───────────────────────────────────────────
  By PR-AUC   : Random Forest             (0.9962)
  By F1-Score : Random Forest             (0.9938)

  Overall best (PR-AUC + F1): Random Forest
